# Analyze Lighthouse

Experimentation with analyzing Lighthouse scores.

In [106]:
import sys
import ast
from pathlib import Path

In [107]:
import pandas as pd
import altair as alt

In [108]:
this_dir = Path("__file__").parent.absolute()

In [150]:
sys.path.append(this_dir.parent)

In [109]:
sys.path.append(str(this_dir.parent / "newshomepages"))

In [153]:
import altair_theme

In [154]:
alt.themes.register('palewire', altair_theme.theme)
alt.themes.enable('palewire')

ThemeRegistry.enable('palewire')

In [110]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [111]:
analysis_dir = this_dir.parent / "_analysis"

Read in the dataframe

In [112]:
df = pd.read_csv(
    extracts_dir / "lighthouse-sample.csv",
    usecols=[
        'handle',
        'file_name',
        'date',
        'performance',
        'accessibility',
        'seo',
        'best_practices',
    ],
    dtype={
        'handle': str,
        'file_name': str,
        'performance': float,
        'accessibility': float,
    },
    parse_dates=["date"]
)

In [113]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11503 entries, 0 to 11502
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   handle          11503 non-null  object        
 1   file_name       11503 non-null  object        
 2   date            11503 non-null  datetime64[ns]
 3   performance     11501 non-null  float64       
 4   accessibility   11503 non-null  float64       
 5   best_practices  11383 non-null  float64       
 6   seo             11503 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 629.2+ KB


Exclude any sites with less than 10 observations

In [114]:
observations_by_site = df.groupby("handle").size().rename("n").reset_index()

In [115]:
not_qualified = observations_by_site[observations_by_site.n < 10]

In [116]:
qualified_df = df[~df.handle.isin(not_qualified.handle)]

Aggregate descriptive statistics for each metric.

In [117]:
agg_df = qualified_df.groupby("handle").agg({
    'performance': ['count', 'median', 'mean', 'min', 'max', 'std'],
    'accessibility': ['count', 'median', 'mean', 'min', 'max', 'std'],
    'seo': ['count', 'median', 'mean', 'min', 'max', 'std'],
    'best_practices': ['count', 'median', 'mean', 'min', 'max', 'std'],
})

In [118]:
agg_df

performance                                        accessibility  \
                   count median      mean   min   max       std         count   
handle                                                                          
100Reporters          13  0.160  0.163077  0.14  0.20  0.018879            13   
11AliveNews           13  0.220  0.220000  0.15  0.28  0.040208            13   
12NewsNow             13  0.170  0.219231  0.08  0.36  0.086068            13   
12khari               13  0.160  0.147692  0.04  0.31  0.069421            13   
13wmaznews            12  0.245  0.266667  0.14  0.38  0.081389            12   
...                  ...    ...       ...   ...   ...       ...           ...   
yorkdispatch          13  0.830  0.819231  0.70  0.96  0.066641            13   
zeitonline            13  0.510  0.500000  0.39  0.58  0.055976            13   
zerohedge             19  0.460  0.438947  0.26  0.54  0.075638            19   
zerohora              19  0.260  0.265263  0.24  0.30  0.016114            19   
zn_ua                 13  0.250  0.248462  0.21  0.30  0.028823            13   

                                     ...       seo                        \
             median      mean   min  ...      mean   min   max       std   
handle                               ...                                   
100Reporters   0.89  0.890000  0.89  ...  0.875385  0.87  0.88  0.005189   
11AliveNews    0.81  0.819231  0.81  ...  0.700000  0.70  0.70  0.000000   
12NewsNow      0.81  0.813077  0.81  ...  0.700000  0.70  0.70  0.000000   
12khari        0.78  0.780000  0.78  ...  0.850000  0.85  0.85  0.000000   
13wmaznews     0.82  0.820000  0.81  ...  0.776667  0.77  0.78  0.004924   
...             ...       ...   ...  ...       ...   ...   ...       ...   
yorkdispatch   0.88  0.892308  0.88  ...  0.935385  0.93  1.00  0.019415   
zeitonline     0.90  0.889231  0.86  ...  0.990000  0.99  0.99  0.000000   
zerohedge      0.95  0.950000  0.95  ...  0.898421  0.89  0.90  0.003746   
zerohora       0.84  0.840000  0.84  ...  0.910000  0.91  0.91  0.000000   
zn_ua          0.66  0.660000  0.66  ...  0.910000  0.91  0.91  0.000000   

             best_practices                                         
                      count median      mean   min   max       std  
handle                                                              
100Reporters             13   0.58  0.580000  0.58  0.58  0.000000  
11AliveNews              13   0.75  0.736923  0.58  0.75  0.047150  
12NewsNow                13   0.83  0.798462  0.58  0.83  0.079146  
12khari                  13   0.83  0.823846  0.75  0.83  0.022188  
13wmaznews               12   0.83  0.776667  0.67  0.83  0.071010  
...                     ...    ...       ...   ...   ...       ...  
yorkdispatch             13   0.92  0.920000  0.92  0.92  0.000000  
zeitonline               13   0.92  0.920000  0.92  0.92  0.000000  
zerohedge                19   0.92  0.915263  0.83  0.92  0.020647  
zerohora                 19   0.92  0.944211  0.83  1.00  0.055908  
zn_ua                    13   0.92  0.906154  0.83  0.92  0.033798  

[776 rows x 24 columns]

Flatten the dataframe

In [119]:
flat_df = agg_df.copy()
flat_df.columns = ['_'.join(col) for col in flat_df.columns]

In [120]:
flat_df.sort_values("performance_count")

,performance_count,performance_median,performance_mean,performance_min,performance_max,performance_std,accessibility_count,accessibility_median,accessibility_mean,accessibility_min,...,seo_mean,seo_min,seo_max,seo_std,best_practices_count,best_practices_median,best_practices_mean,best_practices_min,best_practices_max,best_practices_std
handle,,,,,,,,,,,,,,,,,,,,,
OANN,10,0.120,0.137000,0.10,0.23,0.041379,10,0.87,0.870000,0.87,...,0.910000,0.91,0.91,0.000000,10,0.83,0.830,0.83,0.83,0.000000
oronline,10,0.125,0.120000,0.05,0.21,0.064807,10,0.81,0.802000,0.79,...,0.750000,0.75,0.75,0.000000,10,0.75,0.809,0.75,0.92,0.080478
JustTheNews,10,0.240,0.235000,0.20,0.26,0.018409,10,0.89,0.854000,0.80,...,0.970000,0.97,0.97,0.000000,10,0.75,0.774,0.75,0.83,0.038644
EpochTimes,10,0.225,0.225000,0.18,0.26,0.024152,10,0.86,0.860000,0.84,...,0.767000,0.76,0.83,0.022136,10,0.75,0.750,0.75,0.75,0.000000
dwnews,11,0.160,0.159091,0.15,0.16,0.003015,11,0.89,0.884545,0.87,...,0.860000,0.86,0.86,0.000000,11,0.75,0.750,0.75,0.75,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
foxnews,29,0.180,0.181034,0.14,0.23,0.014229,29,0.83,0.832759,0.83,...,0.820000,0.82,0.82,0.000000,29,0.83,0.830,0.83,0.83,0.000000
ajc,30,0.085,0.099667,0.07,0.14,0.024703,30,0.68,0.680000,0.68,...,0.840000,0.84,0.84,0.000000,30,0.83,0.830,0.83,0.83,0.000000
globeandmail,30,0.200,0.187333,0.10,0.28,0.052387,30,1.00,0.996667,0.98,...,0.978333,0.92,0.99,0.026533,30,0.92,0.920,0.92,0.92,0.000000


Classify the scores

In [121]:
def color_code(val):
    """Return the classification of a metric according to Google's system.
    
    Source: https://developer.chrome.com/docs/lighthouse/performance/performance-scoring/
    """
    if val >= .9:
        return 'green'
    elif val >= .5:
        return 'orange'
    else:
        return 'red'

In [122]:
flat_df['performance_color'] = flat_df.performance_median.apply(color_code)

In [123]:
flat_df['accessibility_color'] = flat_df.accessibility_median.apply(color_code)

Rank the result

In [124]:
flat_df['performance_rank'] = flat_df.performance_median.rank(ascending=False, method="min")

In [125]:
flat_df.sort_values("performance_rank")[[
    'performance_median',
    'performance_rank'
]]

,performance_median,performance_rank
handle,,
tass_agency,1.000,1.0
insideclimate,1.000,1.0
techmeme,0.970,3.0
gigharbornow,0.960,4.0
lobs,0.945,5.0
...,...,...
JournalStarNews,0.020,772.0
BostonHerald,0.020,772.0
portalimprensa,0.020,772.0


Total up the colors

In [126]:
flat_df.performance_color.value_counts()

red       606
orange    161
green       9
Name: performance_color, dtype: int64

In [127]:
flat_df.performance_color.value_counts(normalize=True)

red       0.780928
orange    0.207474
green     0.011598
Name: performance_color, dtype: float64

In [128]:
flat_df.accessibility_color.value_counts()

orange    533
green     239
red         4
Name: accessibility_color, dtype: int64

In [129]:
flat_df.accessibility_color.value_counts(normalize=True)

orange    0.686856
green     0.307990
red       0.005155
Name: accessibility_color, dtype: float64

In [130]:
flat_df.performance_median.describe()

count    776.000000
mean       0.347874
std        0.214348
min        0.010000
25%        0.190000
50%        0.280000
75%        0.475000
max        1.000000
Name: performance_median, dtype: float64

In [131]:
flat_df.accessibility_median.describe()

count    776.000000
mean       0.844407
std        0.099828
min        0.430000
25%        0.780000
50%        0.860000
75%        0.920000
max        1.000000
Name: accessibility_median, dtype: float64

In [171]:
chart_df = (
    qualified_df[qualified_df.handle == 'nytimes']
        .set_index(["handle", "file_name", "date"])
        .stack()
        .reset_index()
        .rename(columns={0: 'value', 'level_3': 'metric'})
)

In [172]:
chart_df['color'] = chart_df.value.apply(color_code)

In [173]:
chart_df.value = chart_df.value * 100

In [174]:
chart_df.metric = chart_df.metric.str.capitalize().str.replace("_" , " ").replace("Seo", "SEO")

In [175]:
chart_df.head()

,handle,file_name,date,metric,value,color
0,nytimes,nytimes-2022-08-10T10:50:26.890757-04:00.light...,2022-08-10,Performance,33.0,red
1,nytimes,nytimes-2022-08-10T10:50:26.890757-04:00.light...,2022-08-10,Accessibility,92.0,green
2,nytimes,nytimes-2022-08-10T10:50:26.890757-04:00.light...,2022-08-10,Best practices,83.0,orange
3,nytimes,nytimes-2022-08-10T10:50:26.890757-04:00.light...,2022-08-10,SEO,91.0,green
4,nytimes,nytimes-2022-08-10T14:14:56.282582-04:00.light...,2022-08-10,Performance,22.0,red


In [184]:
alt.Chart(chart_df).mark_circle(size=50).encode(
    x=alt.X('value:Q', axis=alt.Axis(title=None)),
    y=alt.Y('metric:O', title=None),
    color=alt.Color("color:N", legend=None, scale=alt.Scale(domain=["green", "orange", "red"], range=["green", "orange", "red"])),
    tooltip=["metric", "date", "value"]
).properties(
    title="Lighthouse scores over last 7 days",
    width=500,
    height=200
).configure_axis(
    labelFontSize=14,
    titleFontSize=14
)

alt.Chart(...)